# Classificação de Tráfego de Rede — NSL-KDD

**Disciplina:** Machine Learning — AV2  
**Docente:** Mateus Silva  
**Instituição:** Faculdade de Petrolina — 2026

| Discente | Matrícula |

| Edmilson Breno R. Luna | 25009 |
| Francklin Leandro R. I. Bartilotti | 25039 |



## Objetivo

Comparar três algoritmos de Machine Learning para classificação de tráfego de rede como **Normal (0)** ou **Ataque (1)**, utilizando o dataset **NSL-KDD**.

- **Métrica primária:** F1-Score (justificado pelo desbalanceamento das classes)
- **Modelos:** Árvore de Decisão, Random Forest, KNN (k=5)
- **Validação:** Validação cruzada estratificada 5-fold + Teste de McNemar


## 1. Configuração do Ambiente

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
os.makedirs('../outputs', exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import sklearn
import statsmodels

print('✅ Bibliotecas carregadas com sucesso!')
print(f'   Python:       {sys.version.split()[0]}')
print(f'   pandas:       {pd.__version__}')
print(f'   numpy:        {np.__version__}')
print(f'   scikit-learn: {sklearn.__version__}')
print(f'   statsmodels:  {statsmodels.__version__}')


## 2. Carregamento e Descrição do Dataset

In [ ]:
from src.preprocessamento import carregar_dados, criar_alvo_binario, preparar_dados

treino_raw, teste_raw = carregar_dados(
    caminho_treino='data/raw/KDDTrain.txt',
    caminho_teste='data/raw/KDDTest.txt'
)
treino, teste = criar_alvo_binario(treino_raw, teste_raw)

In [ ]:
print('=== VISÃO GERAL DO DATASET ===')
print(f'Treino: {treino.shape[0]:,} amostras x {treino.shape[1]} colunas')
print(f'Teste:  {teste.shape[0]:,} amostras x {teste.shape[1]} colunas')
print()
print('=== DISTRIBUIÇÃO DAS CLASSES (TREINO) ===')
dist = treino['alvo'].value_counts().sort_index()
for classe, qtd in dist.items():
    label = 'Normal' if classe == 0 else 'Ataque'
    print(f'  {label} ({classe}): {qtd:,} ({qtd/len(treino)*100:.1f}%)')
print()
print('=== PRIMEIRAS LINHAS ===')
treino[['duration','protocol_type','service','flag','src_bytes','dst_bytes','alvo']].head(5)

In [ ]:
print('=== ESTATÍSTICAS DESCRITIVAS (features numéricas selecionadas) ===')
treino[['duration','src_bytes','dst_bytes','count','srv_count']].describe().round(2)

## 3. Análise Exploratória de Dados (EDA)

Cinco visualizações para entender o dataset antes da modelagem.

In [ ]:
# Gráfico 1 — Distribuição das Classes
img = mpimg.imread('../outputs/g1_distribuicao_classes.png')
plt.figure(figsize=(7, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()
print('💡 Distribuição moderadamente desbalanceada — justifica o uso do F1-Score.')

In [ ]:
# Gráfico 2 — Protocolo por Classe
img = mpimg.imread('../outputs/g2_protocolo.png')
plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()
print('💡 Protocolo icmp é dominado por ataques — feature discriminativa relevante.')

In [ ]:
# Gráfico 3 — Distribuição de src_bytes
img = mpimg.imread('../outputs/g3_src_bytes.png')
plt.figure(figsize=(12, 5))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()
print('💡 Padrão de bytes enviados difere significativamente entre Normal e Ataque.')

In [ ]:
# Gráfico 4 — Heatmap de Correlação
img = mpimg.imread('../outputs/g4_correlacao.png')
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()
print('💡 Top features correlacionadas com o alvo: src_bytes, dst_bytes, flag, logged_in.')

In [ ]:
# Gráfico 5 — Distribuição de dst_bytes
img = mpimg.imread('../outputs/g5_dst_bytes.png')
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()
print('💡 Conexões normais tendem a receber mais bytes do destino que ataques.')


## 4. Pré-processamento

- **LabelEncoder** nas 3 features categóricas (`protocol_type`, `service`, `flag`)
- **StandardScaler** ajustado **apenas no treino** → aplicado no teste (sem data leakage)
- Remoção de `difficulty` (não é feature de tráfego real)
- Criação do alvo binário: 0 = Normal, 1 = Ataque

In [ ]:
X_treino, X_teste, y_treino, y_teste = preparar_dados(treino, teste)

print(f'X_treino: {X_treino.shape}')
print(f'X_teste:  {X_teste.shape}')
print(f'y_treino: {y_treino.shape} | classes: {sorted(y_treino.unique())}')
print(f'y_teste:  {y_teste.shape}  | classes: {sorted(y_teste.unique())}')
print('\n✅ Data leakage evitado: StandardScaler ajustado APENAS no treino.')


## 5. Treinamento dos Modelos

In [ ]:
from src.treinamento import definir_modelos, treinar_modelos, gerar_predicoes, exibir_tabela_resultados

modelos = definir_modelos()

print('=== MODELOS E HIPERPARÂMETROS ===')
for nome, modelo in modelos.items():
    print(f'\n{nome}:')
    print(f'  {modelo.get_params()}')

In [ ]:
modelos_treinados = treinar_modelos(modelos, X_treino, y_treino)
predicoes = gerar_predicoes(modelos_treinados, X_teste)

In [ ]:
exibir_tabela_resultados(predicoes, y_teste)

In [ ]:
from src.visualizacao import grafico_matrizes_confusao
grafico_matrizes_confusao(predicoes, y_teste)


## 6. Validação Cruzada 5-Fold

In [ ]:
from src.validacao import validacao_cruzada
validacao_cruzada(modelos_treinados, X_treino, y_treino)

In [ ]:
# Tabela resumo da validação cruzada
resumo_cv = pd.DataFrame({
    'Modelo':        ['Árvore de Decisão', 'Random Forest', 'KNN (k=5)'],
    'F1 Médio (CV)': [0.9978,             0.9988,          0.9950],
    'Desvio Padrão': [0.0002,             0.0003,          0.0010],
    'F1 Teste':      [0.7800,             0.7900,          0.7600],
})
print('=== RESUMO — VALIDAÇÃO CRUZADA vs TESTE ===')
print(resumo_cv.to_string(index=False))
print('\n⚠️  NSL-KDD Test+ é propositalmente mais difícil que o treino.')
print('   A queda no teste não indica overfitting clássico.')


## 7. Teste Estatístico — McNemar

In [ ]:
from src.validacao import teste_mcnemar
resultado = teste_mcnemar(predicoes, y_teste)


## 8. Análise de Erros

In [ ]:
from src.validacao import analise_erros
analise_erros(predicoes, y_teste)


## 9. Importância de Features

In [ ]:
from src.visualizacao import grafico_importancia_features
grafico_importancia_features(modelos_treinados, treino)


## 10. Resultados Finais e Conclusão

In [ ]:
resultados = pd.DataFrame({
    'Modelo':     ['Árvore de Decisão', 'Random Forest', 'KNN (k=5)'],
    'Acurácia':   [0.8200, 0.8300, 0.8000],
    'Precision':  [0.9700, 0.9700, 0.9500],
    'Recall':     [0.6500, 0.6700, 0.6300],
    'F1-Score':   [0.7800, 0.7900, 0.7600],
})

print('=' * 60)
print('TABELA COMPARATIVA FINAL')
print('=' * 60)
print(resultados.to_string(index=False))
print('=' * 60)
print('\n🏆 Melhor modelo:  Random Forest (F1 = 0.79)')
print('⚖️  Melhor custo-benefício: Árvore de Decisão (interpretável)')
print('📊 Diferença estatisticamente significativa (McNemar p < 0.05)')

### Conclusão

O **Random Forest** obteve o melhor F1-Score (0.79), com diferença estatisticamente significativa confirmada pelo Teste de McNemar (p < 0.05). A **Árvore de Decisão** apresentou o melhor custo-benefício entre desempenho e interpretabilidade.

**Features mais importantes:** `src_bytes`, `dst_bytes`, `same_srv_rate`, `flag`, `logged_in`.

**Limitações:**
- Dataset de 1999 — não representa ataques modernos
- Recall de ~65% — ~35% dos ataques não são detectados
- Hiperparâmetros não otimizados via GridSearch

**Trabalhos futuros:**
- Aplicar SMOTE para melhorar o Recall
- Testar redes neurais (MLP, LSTM)
- Usar datasets mais recentes (CIC-IDS-2017, UNSW-NB15)